In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, ICRS, Galactic
import astropy.units as u
import gc

In [ ]:
def read_in_maps(file1,file2):

    hdu  = fits.open(file1)
    map1 = hdu[0].data

    hdu  = fits.open(file2)
    map2 = hdu[0].data   
    
    wcs = WCS(hdu[0].header)

    return map1, map2, wcs

In [ ]:
def make_diff_map(wcs, map1, map2, stokes = 'pi',
                  vmax=20, diff_frac=0.5, l_range = [180,0],b_range = [-10,10]):

    crange = SkyCoord(l_range, b_range, frame=Galactic, unit=(u.deg, u.deg))
    
    fs = 12
    width = 20
    
    height = 4*width*(b_range[1]-b_range[0])/((l_range[0]-l_range[1]))
    print(height)
    
    fig = plt.figure(figsize=(width,height))
    ax1 = fig.add_subplot(411, projection=wcs.celestial)
    ax2 = fig.add_subplot(412, projection=wcs.celestial)
    ax3 = fig.add_subplot(413, projection=wcs.celestial)
    ax4 = fig.add_subplot(414)

    if stokes == 'i':
        vmin = 0
        cmap = 'cubehelix'
    if stokes == 'pi':
        vmin = 0
        cmap = 'viridis'
    if ((stokes == 'q') or (stokes == 'u') or (stokes == 'rm')):
        vmin = -vmax
        cmap = 'RdBu_r'
    if (stokes == 'pa'):
        vmin = -np.pi/2.
        vmax = np.pi/2.
        cmap = 'twilight'

    im1 = ax1.imshow(map1,origin='lower',cmap=cmap,vmin=vmin,vmax=vmax)
    im2 = ax2.imshow(map2,origin='lower',cmap=cmap,vmin=vmin,vmax=vmax)
    im3 = ax3.imshow(map2-map1,origin='lower',cmap='RdBu_r',vmin=-vmax*diff_frac,vmax=vmax*diff_frac)
    ax4.hist(map2.flatten()-map1.flatten(), bins = 101, range=(-vmax*3*diff_frac,vmax*3*diff_frac))
 
    axs = [ax1,ax2,ax3]
    ims = [im1,im2,im3]
    
    for i in range(0,len(axs)):
        axs[i].set_ylim(wcs.world_to_pixel(crange)[1])
        axs[i].set_xlim(wcs.world_to_pixel(crange)[0])
        cbar = fig.colorbar(ims[i], ax=axs[i], shrink=1, pad=0.01)
        axs[i].set_xlabel(' ')
        axs[i].set_ylabel('lat',fontsize=fs)
    ax3.set_xlabel('lon',fontsize=fs)
    ax4.set_yscale('log')

    return

In [ ]:
#file1 = '/srv/data/cgps-gmims_2023/pea'
file1 = '/srv/data/cgps-gmims/gmims_FD/phi_peak_regrd_old.fits'
file2 = '/srv/data/cgps-gmims/gmims_FD/phi_peak_regrd.fits'

map1, map2, wcs = read_in_maps(file1,file2) 

print(map1.shape)
print(map2.shape)

make_diff_map(wcs, map1, map2, stokes = 'rm', vmax=50, diff_frac=0.5, l_range=[180,53], b_range=[-4,6])

del map1,map2,file1,file2
gc.collect()